# 03 · Stream a CommunityForensics-Small subset (unseen generators)

**Accelerator:** none · **Runtime:** 20–60 min (network bound) · **Output dataset name:** `cf-subset`

We never download the 260 GB dataset. The loader streams `OwensLab/CommunityForensics-Small` (split `train`,
bytes in `image_data`, generator in `model_name`, label 1 = fake; verified against the dataset card) and keeps
explicit quotas: 1500 real + 1500 fake, at most 40 fakes per model, bounded scan. Re-running resumes from the
manifest. Licence: CC BY-NC-SA 4.0 (each generated image also carries its source model's licence).

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/kvamsi-iisc/DLVA-Final-Project-Forensic-Verification-of-Marketplace-Listings.git"   # used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
_inp = pathlib.Path("/kaggle/input")   # classic /kaggle/input/<slug> or newer /kaggle/input/datasets/<owner>/<slug>
code_ds = next((p for p in [_inp / "listing-verifier-code", *sorted(_inp.glob("datasets/*/listing-verifier-code"))] if p.exists()), None)
if REPO.exists() and (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "-q"], check=False)     # pick up code fixes in a reused session
elif not REPO.exists():
    if code_ds is not None:
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
print("python", sys.version.split()[0], "| code:", subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stdout[-3000:], pip.stderr[-3000:])          # show WHY before stopping (e.g. a pin without a wheel for this Python)
    raise SystemExit("pip install failed; see output above")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
sys.path.insert(0, str(REPO))
from listing_verifier.kaggle import init, run, attach, optional_attach, merge_fakes, is_attached, input_file, list_attached   # helpers live in the repo: fixes need no re-import
WORK, REPO, DATA, OUT, LOGS = init(WORK)
run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")
print("attached inputs:", list_attached())


In [ ]:
run("scripts/04_stream_forensics.py", "--inspect")

If the printed feature names differ from `image_data` / `label` / `model_name`, pass `--image-col/--label-col/--generator-col`. Then verify by eye that label 1 is fake.

In [ ]:
run("scripts/04_stream_forensics.py", "--n-real", 1500, "--n-fake", 1500, "--per-generator", 40, "--max-scan", 200000)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
df = pd.read_csv(DATA / "forensics" / "forensics_subset.csv"); print(df["label"].value_counts()); print(df["generator"].nunique(), "generators;", df["architecture"].value_counts().head(10) if "architecture" in df else "")
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, (_, r) in zip(axes.ravel(), df.groupby("label").head(6).iterrows()):
    ax.imshow(Image.open(r["path"])); ax.set_title(f"label={r['label']} {str(r['generator'])[:18]}", fontsize=8); ax.axis("off")
plt.tight_layout()

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["data/forensics/forensics_subset.csv"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)